# Trajectory Holiday Context Annotation

This notebook:
1. Fetches trajectories from Fuseki with their first positions
2. Uses Nominatim to get country/city from coordinates
3. Fetches public holidays for the trajectory's year
4. Checks trajectory timestamps against holidays and adds annotations


In [ ]:
import json
import re
import time
from datetime import datetime

import requests
from helpers.fuseki_config import query_endpoint, update_endpoint
from helpers.fuseki_queries import execute_sparql_query, execute_sparql_update

In [ ]:
# API endpoints
nominatim_base = "https://nominatim.openstreetmap.org/reverse"
holidays_base = "https://date.nager.at/api/v3/PublicHolidays"

In [ ]:
# SPARQL query to get trajectories with their first positions
trajectory_query = """
PREFIX datAcron: <http://www.datacron-project.eu/datAcron#>
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX dul: <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>

SELECT ?trajectory ?firstPosition ?point ?lat ?lon ?timestamp WHERE {
  ?trajectory a datAcron:Trajectory .
  ?firstPosition dul:isPartOf ?trajectory ;
                 trajecto:hasPositionIndex 1 ;
                 datAcron:hasGeometry ?point ;
                 trajecto:timestamp ?timestamp .
  ?point datAcron:hasLatitude ?lat ;
         datAcron:hasLongitude ?lon .
}
ORDER BY ?trajectory
"""

print("Executing trajectory query...")
trajectory_results = execute_sparql_query(query_endpoint, trajectory_query)

if trajectory_results and 'results' in trajectory_results:
    trajectories = trajectory_results['results']['bindings']
    print(f"Found {len(trajectories)} trajectories")

    # Show first few results
    for i, traj in enumerate(trajectories[:3]):
        print(f"Trajectory {i+1}:")
        print(f"  URI: {traj['trajectory']['value']}")
        print(f"  Lat/Lon: {traj['lat']['value']}, {traj['lon']['value']}")
        print(f"  Timestamp: {traj['timestamp']['value']}")
else:
    print("Failed to get trajectories")
    trajectories = []

In [ ]:
def get_location_info(lat, lon):
    """Get country and city information from Nominatim."""
    params = {
        'lat': lat,
        'lon': lon,
        'format': 'json',
        'accept-language': 'en',
        'zoom': 5,  # country-level detail only
    }

    headers = {
        "User-Agent": "TrajectoProjectAgent"
    }

    try:
        response = requests.get(nominatim_base, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()

        if 'address' in data:
            address = data['address']
            country = address.get('country', '')
            country_code = address.get('country_code', '').upper()
            city = address.get('city', address.get('town', address.get('village', '')))

            return {
                'country': country,
                'country_code': country_code,
                'city': city,
            }
        else:
            return None

    except requests.exceptions.RequestException as e:
        print(f"Nominatim Error for {lat}, {lon}: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"Nominatim JSON Error for {lat}, {lon}: {e}")
        return None


# Test with the example coordinates
test_location = get_location_info(37.9740118, 23.7358737)
print("Test location lookup:")
print(json.dumps(test_location, indent=2) if test_location else "Failed")

In [ ]:
def get_public_holidays(year, country_code):
    """Get public holidays for a specific year and country."""
    url = f"{holidays_base}/{year}/{country_code}"

    try:
        response = requests.get(url)
        response.raise_for_status()
        holidays = response.json()

        # Convert to a set of date strings for easy lookup
        holiday_dates = {holiday['date'] for holiday in holidays}
        return holiday_dates, holidays

    except requests.exceptions.RequestException as e:
        print(f"Holidays API Error for {year}/{country_code}: {e}")
        return set(), []
    except json.JSONDecodeError as e:
        print(f"Holidays JSON Error for {year}/{country_code}: {e}")
        return set(), []


# Test with Greece 2025
test_holidays, test_holiday_list = get_public_holidays(2025, 'GR')
print(f"Found {len(test_holidays)} holidays for Greece 2025:")
print(list(test_holidays)[:5])  # Show first 5

In [ ]:
def extract_date_from_timestamp(timestamp_str):
    """Extract date (YYYY-MM-DD) and year from timestamp string."""
    # Handle various timestamp formats
    try:
        # Try parsing ISO format first
        if 'T' in timestamp_str:
            dt = datetime.fromisoformat(timestamp_str.replace('Z', '+00:00'))
        else:
            # Try other common formats
            dt = datetime.strptime(timestamp_str, '%Y-%m-%d %H:%M:%S')

        return dt.strftime('%Y-%m-%d'), dt.year
    except ValueError:
        # Try to extract date with regex if parsing fails
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', timestamp_str)
        if match:
            year, month, day = match.groups()
            date_str = f"{year}-{month}-{day}"
            return date_str, int(year)

        print(f"Could not parse timestamp: {timestamp_str}")
        return None, None


# Test with sample timestamp
test_ts = "2024-03-25T10:30:00Z"
date, year = extract_date_from_timestamp(test_ts)
print(f"Timestamp: {test_ts} -> Date: {date}, Year: {year}")

In [ ]:
# Process trajectories and check for holidays
holiday_trajectories = []
location_cache = {}  # Cache location lookups (1-decimal-place grid)
holiday_cache = {}   # Cache holiday lookups

print(f"Processing {len(trajectories)} trajectories...")

api_calls = 0
for i, traj_data in enumerate(trajectories):
    trajectory_uri = traj_data['trajectory']['value']
    lat = float(traj_data['lat']['value'])
    lon = float(traj_data['lon']['value'])
    timestamp = traj_data['timestamp']['value']

    # Extract date and year from timestamp
    date_str, year = extract_date_from_timestamp(timestamp)
    if not date_str:
        continue

    # Get location info — cache at 1 decimal place (~11km grid, plenty for country)
    location_key = f"{lat:.1f},{lon:.1f}"
    if location_key not in location_cache:
        if api_calls > 0:
            time.sleep(1)  # Nominatim rate limit: 1 req/s
        location_cache[location_key] = get_location_info(lat, lon)
        api_calls += 1

    location_info = location_cache[location_key]
    if not location_info:
        continue

    country_code = location_info['country_code']

    # Get holidays for this year and country (with caching)
    holiday_key = f"{year}_{country_code}"
    if holiday_key not in holiday_cache:
        holiday_dates, holiday_list = get_public_holidays(year, country_code)
        holiday_cache[holiday_key] = (holiday_dates, holiday_list)
    else:
        holiday_dates, _ = holiday_cache[holiday_key]

    # Check if trajectory date is a holiday
    if date_str in holiday_dates:
        holiday_trajectories.append({
            'trajectory_uri': trajectory_uri,
            'date': date_str,
            'country': location_info['country'],
            'country_code': country_code,
            'city': location_info['city'],
            'timestamp': timestamp
        })

    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(trajectories)} processed, {api_calls} API calls, "
              f"{len(holiday_trajectories)} holidays found")

print(f"\nDone: {len(trajectories)} trajectories, {api_calls} Nominatim lookups, "
      f"{len(holiday_trajectories)} on holidays")
for ht in holiday_trajectories:
    print(f"  {ht['trajectory_uri']} on {ht['date']} in {ht['city']}, {ht['country']}")

In [ ]:
# Add holiday annotations to trajectories


def add_holiday_annotation(trajectory_uri):
    """Add holiday context annotation to a trajectory."""
    update_query = f"""
    PREFIX trajecto: <http://w3id.org/trajecto#>
    PREFIX context: <http://w3id.org/trajecto/context#>

    INSERT DATA {{
        <{trajectory_uri}> context:hasTemporalContext context:SpecialPeriod .
    }}
    """

    return execute_sparql_update(update_endpoint, update_query)


# Apply annotations
print(f"Adding holiday annotations to {len(holiday_trajectories)} trajectories...")

success_count = 0
for ht in holiday_trajectories:
    trajectory_uri = ht['trajectory_uri']
    print(f"Annotating {trajectory_uri}...")

    if add_holiday_annotation(trajectory_uri):
        print("  ✓ Successfully added holiday annotation")
        success_count += 1
    else:
        print("  ✗ Failed to add holiday annotation")

print(f"\nSuccessfully annotated {success_count}/{len(holiday_trajectories)} trajectories")

In [ ]:
# Verify annotations were added
verify_query = """
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX context: <http://w3id.org/trajecto/context#>

SELECT ?trajectory WHERE {
    ?trajectory context:hasTemporalContext context:SpecialPeriod .
}
"""

print("Verifying holiday annotations...")
verify_results = execute_sparql_query(query_endpoint, verify_query)

if verify_results and 'results' in verify_results:
    annotated_trajectories = verify_results['results']['bindings']
    print(f"Found {len(annotated_trajectories)} trajectories with holiday annotations:")

    for traj in annotated_trajectories:
        print(f"  - {traj['trajectory']['value']}")
else:
    print("No trajectories with holiday annotations found")

In [ ]:
# Summary statistics
print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"Total trajectories processed: {len(trajectories)}")
print(f"Trajectories on holidays: {len(holiday_trajectories)}")
print(f"Successfully annotated: {success_count}")
print(f"Unique countries found: {len(set(ht['country'] for ht in holiday_trajectories))}")

if holiday_trajectories:
    countries = [ht['country'] for ht in holiday_trajectories]
    from collections import Counter
    country_counts = Counter(countries)
    print("\nCountry breakdown:")
    for country, count in country_counts.items():
        print(f"  {country}: {count} trajectories")

print(f"\nLocation lookups cached: {len(location_cache)}")
print(f"Holiday lookups cached: {len(holiday_cache)}")

Possible extension: a certainty level that decreases with the distance from the holiday date.


In [ ]:
# Validate KG state against stage-03-enrichment SHACL shapes
from helpers.shacl_validator import validate_before_import, construct_validation_graph

proposed = construct_validation_graph('stage-03-enrichment')
print(f"Fetched {len(proposed)} triples for validation")
report = validate_before_import(proposed, 'stage-03-enrichment', halt_on_violations=True)